# Cyberbullying Detector

This notebook shows a step-by-step pipeline for detecting cyber-aggressive language in social media text.

The project uses a TF-IDF + logistic regression baseline and keeps the process explainable and beginner-friendly.


In [ ]:
from pathlib import Path

dataset_path = Path('../data/Dataset_for_Detection_of_Cyber-Trolls_1.json')
print(f"Dataset exists: {dataset_path.exists()}")


In [ ]:
import html
import pandas as pd

raw = pd.read_json(dataset_path, lines=True)
def first_label(annotation):
    if not isinstance(annotation, dict):
        return None
    labels = annotation.get("label")
    if not isinstance(labels, list) or not labels:
        return None
    return str(labels[0]).strip()

data = pd.DataFrame({
    "text": raw.get("content"),
    "label": raw.get("annotation", pd.Series(index=raw.index, dtype=object)).map(first_label),
})
data["text"] = data["text"].fillna("").astype(str).map(lambda text: " ".join(html.unescape(text).split()))
data["label"] = data["label"].astype("string").str.strip()
data = data[data["text"].str.strip().ne("") & data["label"].isin(["0", "1"])].copy()
data["label"] = data["label"].astype(int)
data.head()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["label"],
    test_size=0.20,
    random_state=42,
    stratify=data["label"],
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=1200, class_weight="balanced", random_state=42)),
])
model.fit(X_train, y_train)
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import joblib

predictions = model.predict(X_test)
print(classification_report(
    y_test,
    predictions,
    labels=[0, 1],
    target_names=["non-aggressive", "cyber-aggressive"],
    zero_division=0,
))
print(confusion_matrix(y_test, predictions, labels=[0, 1]))
joblib.dump(model, "../cyberbully_baseline.joblib")
print("Saved model to ../cyberbully_baseline.joblib")
